In [1]:
# Imports
import sys
import os

# Add the upstream directory to sys.path
upstream_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if upstream_dir not in sys.path:
    sys.path.insert(0, upstream_dir)

# Now you can import the module
from opentrons_api import ot2_api
from microtissue_manipulator import core, utils, camera
import numpy as np 
import cv2
import time
import json
import keyboard
# from pynput import keyboard
import paths
import matplotlib.pyplot as plt
import requests
import datetime
import threading
import queue
import string
import pandas as pd
import multiprocessing as mp
from dataclasses import dataclass, fields, asdict, MISSING
from typing import get_type_hints, get_origin, get_args, Tuple, List, Dict, Any, Union
from ultralytics import YOLO
from enum import Enum

a

# Connect camera

In [4]:
cam_manager = camera.CameraManagerWindows()
print("Connected cameras:")
print(cam_manager.list_devices())

microscope_cam = camera.open_capture('microscope_cam', cam_manager=cam_manager)

Connected cameras:
['Digital Microscope', 'HD USB CAMERA']


In [5]:
microscope_cam.read()

(True,
 array([[[10,  1,  0],
         [16, 10,  8],
         [ 0,  0,  4],
         ...,
         [ 1,  0,  1],
         [ 1,  0,  1],
         [ 1,  0,  1]],
 
        [[10,  1,  0],
         [17, 10,  8],
         [ 0,  0,  4],
         ...,
         [ 1,  0,  1],
         [ 1,  0,  1],
         [ 1,  0,  1]],
 
        [[15,  4,  0],
         [15,  7,  5],
         [ 1,  0,  5],
         ...,
         [ 1,  0,  1],
         [ 1,  0,  1],
         [ 1,  0,  1]],
 
        ...,
 
        [[ 1,  0,  1],
         [ 1,  0,  1],
         [ 1,  0,  1],
         ...,
         [ 0,  0,  1],
         [ 3,  0,  3],
         [ 4,  0,  4]],
 
        [[ 1,  0,  1],
         [ 1,  0,  1],
         [ 1,  0,  1],
         ...,
         [ 0,  0,  1],
         [ 3,  0,  3],
         [ 4,  0,  4]],
 
        [[ 1,  0,  1],
         [ 1,  0,  1],
         [ 1,  0,  1],
         ...,
         [ 0,  0,  1],
         [ 3,  0,  3],
         [ 4,  0,  4]]], dtype=uint8))

# Init robot

In [6]:
# Connect the robot to the computer and this notebook
openapi = ot2_api.OpentronsAPI()

In [7]:
# Set the offset if the platform is used
openapi.add_slot_offsets([5,8,9], (0,0,64.2))

In [9]:
# Use the light control to see if the robot is connected as a sanity check
openapi.toggle_lights()

<Response [200]>

In [10]:
# Always do once after robot was just turned on
openapi.home_robot()

Request status:
<Response [200]>
{
  "message": "Homing robot."
}


<Response [200]>

In [11]:
# Use to restore labware and general run information after the notebook crashes
r = openapi.get_run_info()

Total number of runs: 20
Current run ID: ce105718-8a60-4c9f-9546-503f33059d77
Current run status: idle


In [8]:
# Do after first launch
openapi.create_run()

Request status:
<Response [201]>
{
  "data": {
    "id": "ce105718-8a60-4c9f-9546-503f33059d77",
    "ok": true,
    "createdAt": "2025-06-11T20:13:31.526252Z",
    "status": "idle",
    "current": true,
    "actions": [],
    "errors": [],
    "hasEverEnteredErrorRecovery": false,
    "pipettes": [],
    "modules": [],
    "labware": [],
    "liquids": [],
    "liquidClasses": [],
    "labwareOffsets": [],
    "runTimeParameters": [],
    "outputFileIds": []
  }
}


<Response [201]>

In [9]:
# Let the robot know that it has the P300 pipette
openapi.load_pipette()

Request status:
<Response [201]>
{
  "data": {
    "id": "b537ebdd-2c62-4232-bbc2-4c4dc2a146fb",
    "createdAt": "2025-06-11T20:13:35.599311Z",
    "commandType": "loadPipette",
    "key": "b537ebdd-2c62-4232-bbc2-4c4dc2a146fb",
    "status": "succeeded",
    "params": {
      "pipetteName": "p300_single_gen2",
      "mount": "left"
    },
    "result": {
      "pipetteId": "b5e6284d-5b31-4762-af0e-972624a3b749"
    },
    "startedAt": "2025-06-11T20:13:35.602547Z",
    "completedAt": "2025-06-11T20:13:37.552622Z",
    "intent": "setup",
    "notes": []
  }
}


<Response [201]>

In [ ]:
openapi.move_labware(openapi.labware_dct['5'], 'offDeck')

# Labware declaration

### Opentrons 300 ul

In [11]:
#Define a tip rack. This is the default tip rack for the robot.
TIP_RACK = "opentrons_96_tiprack_300ul"
#Load the tip rack. Slot = 1 by default.
r = openapi.load_labware(TIP_RACK, 11)

Labware ID:
6095ae59-a4d4-40be-82b2-b4d2ba3160f5



In [12]:
r = openapi.pick_up_tip(openapi.labware_dct['11'], "A2")

In [10]:
custom_labware_path = os.path.join(paths.BASE_DIR,'protocols','vwr_96_tiprack_200ul_xl.json')
with open(custom_labware_path, 'r') as json_file:
    custom_labware = json.load(json_file)

command_dict = {
            "data": custom_labware
        }

command_payload = json.dumps(command_dict)

url = openapi.get_url('runs')+ f'/{openapi.run_id}/'+ 'labware_definitions'
r = requests.post(url = url, headers = openapi.HEADERS, params = {"waitUntilComplete": True}, data = command_payload)
#Define a tip rack. This is the default tip rack for the robot.
TIP_RACK = "vwr_96_tiprack_200ul_xl"
#Load the tip rack. Slot = 1 by default.
openapi.load_labware(TIP_RACK, 10, namespace='custom_beta',verbose=True)

Labware ID:
85c7d9db-bc87-422b-af68-a0cc02b9eaa1



<Response [201]>

In [20]:
r = openapi.drop_tip_in_place()

In [11]:
r = openapi.pick_up_tip(openapi.labware_dct['10'], "F1")


### Well plate

In [12]:
WELL_PLATE = "corning_96_wellplate_360ul_flat"
# WELL_PLATE = "corning_24_wellplate_3.4ml_flat"
# WELL_PLATE = "corning_384_wellplate_112ul_flat"
r = openapi.load_labware(WELL_PLATE, 5, namespace='opentrons',verbose=True)

Offset (0, 0, 64.2) added to run for corning_96_wellplate_360ul_flat in slot 5.
Labware URI:
opentrons/corning_96_wellplate_360ul_flat/1

Check offset before using ...
Labware ID:
712e6a4b-a6f8-4885-a353-b87fab9fe1a3



# Slice picking

In [13]:
openapi.toggle_lights()

<Response [200]>

In [13]:
openapi.retract_axis('leftZ')
openapi.move_to_well(openapi.labware_dct['5'], 'A1', well_location='top', offset=(0,0,1))

<Response [201]>

In [16]:
openapi.retract_axis('leftZ')
openapi.move_to_coordinates((300, 135, 110), min_z_height=1, verbose=False)

<Response [201]>

In [ ]:
def on_mouse_click(event, cX, cY, flags, param):
    global X_init, Y_init, X, Y, target_x, target_y

    if event == cv2.EVENT_RBUTTONDOWN:
        x, y, _ = openapi.get_position(verbose=False)[0].values()
        openapi.retract_axis('leftZ')
        openapi.move_to_coordinates((300, 135, 110), min_z_height=1, verbose=False)

    if event == cv2.EVENT_MOUSEWHEEL:
        # List of well names in 96-well plate (A1-H12)
        if not hasattr(on_mouse_click, "well_index"):
            on_mouse_click.well_index = 0
        if not hasattr(on_mouse_click, "well_names"):
            rows = "ABCDEFGH"
            cols = range(1, 13)
            on_mouse_click.well_names = [f"{row}{col}" for row in rows for col in cols]
        num_wells = len(on_mouse_click.well_names)
        # event flags: positive for scroll up, negative for scroll down
        if flags > 0:
            on_mouse_click.well_index = (on_mouse_click.well_index + 1) % num_wells
        else:
            on_mouse_click.well_index = (on_mouse_click.well_index - 1) % num_wells
        current_well = on_mouse_click.well_names[on_mouse_click.well_index]
        # print(f"Selected well: {current_well}")


# Create an instance of the ManualRobotMovement class
manual_movement = utils.ManualRobotMovement(openapi)
cv2.namedWindow("video", cv2.WINDOW_NORMAL)
cv2.resizeWindow("video", 1348, 1011)
# cv2.resizeWindow("video", 1050, 1348)
cv2.setMouseCallback("video", on_mouse_click)

target_x, target_y = 0, 0

dish_bottom = 70# - 11.5
pickup_offset = 0.0 #0.6
flow_rate = 280
volume = 10

x_asp, y_asp, z_asp = (300, 135, 110)

while True:
    ret, frame = microscope_cam.read()
    frame = cv2.flip(frame, 1)
    cv2.circle(frame, (target_x, target_y), 3, (0, 0, 255), -1)
    x, y, z = openapi.get_position(verbose=False)[0].values()
    cv2.putText(frame, f"Robot coords: ({x:.2f}, {y:.2f}, {z:.2f})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Step size: {manual_movement.step} mm", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Flow rate: {flow_rate} uL/s", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f"Volume: {volume} uL", (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    if hasattr(on_mouse_click, "well_names") and hasattr(on_mouse_click, "well_index"):
        current_well = on_mouse_click.well_names[on_mouse_click.well_index]
        cv2.putText(frame, f"Current well: {current_well}", (10, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("video", frame)

    # Draw a point at the center of the screen
    key_pressed = cv2.waitKey(1)

    if key_pressed == ord('q'):
        keyboard.unhook_all()
        break

    elif key_pressed == ord('d'):
        openapi.dispense_in_place(flow_rate = flow_rate, volume = volume)

    elif key_pressed == ord('b'):
        openapi.blow_out_in_place(50)

    elif key_pressed == ord('a'):
        x_asp, y_asp, z_asp = openapi.get_position(verbose=False)[0].values()
        openapi.aspirate_in_place(flow_rate = flow_rate, volume = volume, verbose=True)

    elif key_pressed == ord('r'):
        openapi.retract_axis('leftZ')
        openapi.move_to_coordinates((x_asp, y_asp, z_asp + 10), min_z_height=1, verbose=False)

    elif key_pressed == ord('g'):
        if current_well:
            openapi.retract_axis('leftZ')
            openapi.move_to_well(openapi.labware_dct['5'], well_name=current_well, well_location='top', offset = (-0.5, 0, 1), min_z_height=70, verbose=False)
# CREATE folder first, will not save photos unless folder already exists
    elif key_pressed == ord('s'):
        save_dir = "../outputs/images/2026-01-08_PY8119_Biopsy_Core_400um/"
        if not os.path.exists(save_dir):
            os.makedirs(save_dir)
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"frame_{timestamp}.png"
        filepath = os.path.join(save_dir, filename)
        cv2.imwrite(filepath, frame)
        print(f"Frame saved as {filepath}")
# Code directory to make new folder 

cv2.destroyAllWindows()

Frame saved as ../outputs/images/2026-01-08_PY8119_Biopsy_Core_400um/frame_20260109_142241.png
Frame saved as ../outputs/images/2026-01-08_PY8119_Biopsy_Core_400um/frame_20260109_142316.png
Frame saved as ../outputs/images/2026-01-08_PY8119_Biopsy_Core_400um/frame_20260109_142353.png
Request status:
<Response [201]>
{
  "data": {
    "id": "951792c9-f28b-4257-953d-0826f514e7ac",
    "createdAt": "2025-06-11T23:27:54.152858Z",
    "commandType": "aspirateInPlace",
    "key": "951792c9-f28b-4257-953d-0826f514e7ac",
    "status": "succeeded",
    "params": {
      "flowRate": 280.0,
      "volume": 10.0,
      "pipetteId": "b5e6284d-5b31-4762-af0e-972624a3b749"
    },
    "result": {
      "volume": 10.0
    },
    "startedAt": "2025-06-11T23:27:54.155445Z",
    "completedAt": "2025-06-11T23:27:54.581616Z",
    "intent": "setup",
    "notes": []
  }
}
Request status:
<Response [201]>
{
  "data": {
    "id": "5f545b7c-de2c-4469-8e01-8012676e2eb4",
    "createdAt": "2025-06-11T23:28:02.3969

ConnectionError: ('Connection aborted.', ConnectionAbortedError(10053, 'An established connection was aborted by the software in your host machine', None, 10053, None))

: 

In [11]:
openapi.labware_dct['5']

'0f064279-1149-415b-b598-609b23c1329f'